In [ ]:
import scanpy as sc
import anndata as ad
import scipy

from rpy2.robjects import pandas2ri


import anndata2ri

pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

In [ ]:
adata = sc.read_h5ad("./Data/RNA_ATAC/BMMC/BMMC.h5ad")

adata_gex = adata[:, adata.var['modality'] == "Gene Expression"]
adata_atac = adata[:, adata.var['modality'] == "Peaks"]
print(adata_gex)
print(adata_atac)

In [ ]:
%%R
suppressPackageStartupMessages({
    library(SingleCellExperiment)
    library(Seurat)
    library(Signac)
})

In [4]:
adata_ = ad.AnnData(adata_gex.X)
adata_.obs_names = adata_gex.obs_names
adata_.var_names = adata_gex.var_names
adata_.obs['cell_type'] = adata_gex.obs['cell_type']
adata_.obs['batch'] = adata_gex.obs['batch']

In [ ]:
%%R -i adata_
rna = as.Seurat(adata_, counts='X', data=NULL)
rna <- RenameAssays(rna, originalexp="RNA")
rna.list <- SplitObject(rna, split.by = "batch")
rna.list <- lapply(X = rna.list, FUN = SCTransform, variable.features.n = 1000)
features <- SelectIntegrationFeatures(object.list = rna.list, nfeatures = 1000)
rna.list <- PrepSCTIntegration(object.list = rna.list, anchor.features = features)
anchors <- FindIntegrationAnchors(object.list = rna.list, normalization.method = "SCT", anchor.features = features)
integrated <- IntegrateData(anchorset = anchors, normalization.method = "SCT")
integrated <- RunPCA(integrated)


In [6]:
adata_ = ad.AnnData(adata_atac.X)
adata_.obs_names = adata_atac.obs_names
adata_.var_names = adata_atac.var_names
adata_.obs['cell_type'] = adata_atac.obs['cell_type']
adata_.obs['batch'] = adata_atac.obs['batch']

In [ ]:
%%R -i adata_
atac = as.Seurat(adata_, counts='X', data=NULL)
atac <- RenameAssays(atac, originalexp ='ATAC')
atac.list <- SplitObject(atac, split.by = "batch")

atac.list <- lapply(X = atac.list, FUN = function(X) {
  DefaultAssay(X) <- "ATAC"
  X <- FindTopFeatures(X, min.cutoff = 10)
  X <- RunTFIDF(X)
  X <- RunSVD(X)
})
combined <- merge(x = atac.list[[1]], y= atac.list[2:length(atac.list)] )

combined <- FindTopFeatures(combined, min.cutoff = 10)
combined <- RunTFIDF(combined)
combined <- RunSVD(combined)

integration.anchors <- FindIntegrationAnchors(
  object.list = atac.list,
  anchor.features = rownames(combined),
  reduction = "rlsi",
  dims = 2:30
)

# integrate LSI embeddings
integrated_atac <- IntegrateEmbeddings(
  anchorset = integration.anchors,
  reductions = combined[["lsi"]],
  new.reduction.name = "integrated_lsi",
  dims.to.integrate = 1:30
)

In [8]:
%%R
integrated[['ATAC']] = integrated_atac[['ATAC']]
integrated[["Iatac"]] <- integrated_atac[["integrated_lsi"]]
integrated <- FindMultiModalNeighbors(integrated, reduction.list = list("pca", "Iatac"), 
                                      dims.list = list(1:50, 1:30), modality.weight.name = "RNA.weight")
# 
integrated <- RunSPCA(integrated, assay = 'integrated', graph = 'wsnn', npcs = 20)


  |                                                  | 0 % ~calculating   |+++++++++++++++++++++++++                         | 50% ~26s           |++++++++++++++++++++++++++++++++++++++++++++++++++| 100% elapsed=50s  
  |                                                  | 0 % ~calculating   |+++++++++++++++++++++++++                         | 50% ~05s           |++++++++++++++++++++++++++++++++++++++++++++++++++| 100% elapsed=09s  
  |                                                  | 0 % ~calculating   |+++++++++++++++++++++++++                         | 50% ~01m 19s       |++++++++++++++++++++++++++++++++++++++++++++++++++| 100% elapsed=02m 47s
  |                                                  | 0 % ~calculating   |+++++++++++++++++++++++++                         | 50% ~12s           |++++++++++++++++++++++++++++++++++++++++++++++++++| 100% elapsed=19s  


Calculating cell-specific modality weights
Finding 20 nearest neighbors for each modality.
Calculating kernel bandwidths
Finding multimodal neighbors
Constructing multimodal KNN graph
Constructing multimodal SNN graph
Computing sPCA transformation
In addition: Warning message:
In FindMultiModalNeighbors(integrated, reduction.list = list("pca",  :
  The number of provided modality.weight.name is not equal to the number of modalities. integrated.weight ATAC.weight are used to store the modality weights


In [9]:
%%R -o spca
spca = Embeddings(object = integrated[["spca"]])

In [10]:
adata = sc.AnnData(spca)
adata.obs = adata_.obs
adata

AnnData object with n_obs × n_vars = 69249 × 20
    obs: 'cell_type', 'batch'

In [11]:
%%R -o wnn
wnn <- as.data.frame(summary(integrated@graphs$wknn))

In [ ]:
wnn['i'] = wnn['i'] - 1
wnn['j'] = wnn['j'] - 1
adata.obsp['wnn_connectivities'] = scipy.sparse.coo_matrix((wnn['x'], (wnn['i'], wnn['j'])))
adata.obsp['wnn_connectivities'] = scipy.sparse.csr_matrix(adata.obsp['wnn_connectivities'])
adata.write('Seurat_BMMC.h5ad')